# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd
import numpy as np

dataset = pd.read_csv('work/outputs/dataset.csv')

# The rule only ever looks at *this* window's behavior -- never trend_direction/trend_pct
# or the March/April impression columns, which are the ML-04 label-leak list.
RULE_THRESHOLDS = {
    'visible_min': dataset['impressions_90d'].quantile(0.50),   # "gets real search visibility"
    'poor_rank_min': dataset['avg_position_90d'].quantile(0.75), # bottom quartile = worst ranks
    'low_consistency_max': dataset['active_days_90d'].quantile(0.25),
    'low_ctr_max': dataset['ctr_90d'].quantile(0.25),
}
ga4 = dataset[dataset['has_ga4_data'] == 1].copy()
ga4['engagement_ratio'] = ga4['engaged_sessions_90d'] / ga4['sessions_90d'].replace(0, np.nan)
RULE_THRESHOLDS['low_engagement_max'] = ga4['engagement_ratio'].quantile(0.25)

print("Thresholds are defined as quantiles of THIS dataset, not hardcoded numbers --")
print("that keeps the rule readable ('bottom quartile') without guessing an absolute cutoff:")
for k, v in RULE_THRESHOLDS.items():
    print(f"  {k:22s} = {v:.3f}")


Thresholds are defined as quantiles of THIS dataset, not hardcoded numbers --
that keeps the rule readable ('bottom quartile') without guessing an absolute cutoff:
  visible_min            = 1467.000
  poor_rank_min          = 17.321
  low_consistency_max    = 41.000
  low_ctr_max            = 0.000
  low_engagement_max     = 0.000


## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth reviewing first if it gets real search
visibility (at least the median `impressions_90d` for this dataset) **and** shows at least
one warning sign: it's already trending down within the 90-day feature window, it ranks
poorly, its search visibility has been inconsistent, its click-through is weak for how much
it's shown, or (when GA4 is available) people who land on it barely engage. A page nobody
sees is never worth reviewing first, no matter how bad its other numbers look -- see the
multiplicative score in section 2.

**Reason codes it can output** (a page can carry more than one):

| Reason code | Fires when |
|---|---|
| `visible_declining_momentum` | visible, and already trending down in-window (`momentum_pct < 0`, restricted to `has_momentum == 1`) |
| `visible_poor_rank` | visible, and `avg_position_90d` is in the worst quartile |
| `inconsistent_visibility` | visible, and `active_days_90d` is in the lowest quartile |
| `low_ctr_visible_page` | visible, and `ctr_90d` is in the lowest quartile |
| `low_engagement_visible_page` | visible, has GA4 data, and engaged/total session ratio is in the lowest quartile |
| `general_review_candidate` | visible, but none of the above fired -- still worth a light look |

This mirrors `scripts/02_baseline_score.py`'s approach (visibility gates everything, plain
conditions, no fitted weights) rebuilt on this dataset's own columns -- the starter-CSV
script uses `word_count`/`engagement_rate`/`days_since_last_update`, none of which exist in
`work/outputs/dataset.csv`.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
def percentile_rank(s):
    return s.rank(pct=True, method='average')

def reason_codes(row, th):
    visible = row['impressions_90d'] >= th['visible_min']
    reasons = []
    if not visible:
        return ['not_visible_low_priority']
    if row['has_momentum'] == 1 and row['momentum_pct'] < 0:
        reasons.append('visible_declining_momentum')
    if row['avg_position_90d'] >= th['poor_rank_min']:
        reasons.append('visible_poor_rank')
    if row['active_days_90d'] <= th['low_consistency_max']:
        reasons.append('inconsistent_visibility')
    if row['ctr_90d'] <= th['low_ctr_max']:
        reasons.append('low_ctr_visible_page')
    if row['has_ga4_data'] == 1 and row['sessions_90d'] > 0:
        ratio = row['engaged_sessions_90d'] / row['sessions_90d']
        if ratio <= th['low_engagement_max']:
            reasons.append('low_engagement_visible_page')
    if not reasons:
        reasons.append('general_review_candidate')
    return reasons

def suggested_action(reasons):
    reasons = set(reasons)
    if 'not_visible_low_priority' in reasons:
        return 'monitor'
    if 'low_ctr_visible_page' in reasons or 'low_engagement_visible_page' in reasons:
        return 'refresh_metadata_or_content'
    if 'visible_declining_momentum' in reasons or 'visible_poor_rank' in reasons:
        return 'review'
    if 'inconsistent_visibility' in reasons:
        return 'monitor'
    return 'monitor'

dataset['reason_code_list'] = dataset.apply(lambda r: reason_codes(r, RULE_THRESHOLDS), axis=1)
dataset['reason_codes'] = dataset['reason_code_list'].apply(lambda r: '|'.join(r))
dataset['suggested_action_baseline'] = dataset['reason_code_list'].apply(suggested_action)

# Score: visibility GATES the risk composite -- an invisible page scores near zero
# regardless of its other numbers, same "stale * visible" spirit as scripts/02_baseline_score.py.
visibility_score = percentile_rank(np.log1p(dataset['impressions_90d']))

momentum_risk_score = pd.Series(0.5, index=dataset.index)  # neutral default: unknown, not "safe"
has_mom_mask = dataset['has_momentum'] == 1
momentum_risk_score.loc[has_mom_mask] = percentile_rank(-dataset.loc[has_mom_mask, 'momentum_pct'])
# has_momentum == 0 rows had zero baseline impressions (Jan-Feb) -- there's no real trend to
# read, so they get the neutral 0.5 rather than 0 (which would wrongly look "safe") or 1.

position_risk_score = percentile_rank(dataset['avg_position_90d'])
consistency_risk_score = 1 - percentile_rank(dataset['active_days_90d'])

risk_composite = (
    0.45 * momentum_risk_score
    + 0.30 * position_risk_score
    + 0.25 * consistency_risk_score
)

dataset['baseline_action_score'] = (visibility_score * risk_composite).clip(0, 1)
dataset['baseline_rank'] = dataset['baseline_action_score'].rank(method='first', ascending=False).astype(int)

queue = dataset.sort_values('baseline_rank')[[
    'baseline_rank', 'client_hash_id', 'content_hash_id', 'baseline_action_score',
    'reason_codes', 'suggested_action_baseline', 'is_declining_label',
    'impressions_90d', 'avg_position_90d', 'momentum_pct', 'active_days_90d', 'ctr_90d',
]]

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = dataset['is_declining_label'].mean()
print(f"\nBase rate (declining share of ALL rows): {base_rate:.3f}")
for k in (10, 25, 50):
    p = precision_at_k(dataset['baseline_action_score'].values, dataset['is_declining_label'].values, k)
    lift = p / base_rate if base_rate > 0 else float('nan')
    print(f"Precision@{k}: {p:.3f}   (base rate {base_rate:.3f}, {lift:.1f}x lift)")


Wrote 103,691 rows to work/outputs/baseline_action_score.csv

Base rate (declining share of ALL rows): 0.375
Precision@10: 0.600   (base rate 0.375, 1.6x lift)
Precision@25: 0.480   (base rate 0.375, 1.3x lift)
Precision@50: 0.540   (base rate 0.375, 1.4x lift)


## 2. Build the ranked queue (writes the CSV)

The score is **visibility x risk**, not visibility + risk: `visibility_score` (percentile
rank of `log1p(impressions_90d)`, heavy-tail-safe per the ML-06 audit) multiplies a
`risk_composite` built from the three signals ML-06 tested (in-window momentum, rank
position, visibility consistency), weighted 0.45 / 0.30 / 0.25 by hand -- no fitted weights,
per `skills/building-baselines/SKILL.md`. A page with zero visibility scores near zero no
matter how risky its other numbers look, the same "stale * visible" logic as
`scripts/02_baseline_score.py`.

`has_momentum == 0` rows (no Jan-Feb baseline to compute a real trend from) get a **neutral**
momentum-risk score of 0.5, not 0 or 1 -- treating "unknown" as "safe" or "risky" would
quietly inject a fabricated signal, the exact trap `skills/flyrank-data/SKILL.md` warns about
for structural flags.

Precision@10/25/50 and the base rate are printed above, computed on the same rows and the
same `is_declining_label` the Week-5 model will be evaluated against later.

**Precision@10 = 0.600 vs. a base rate of 0.375 -- a 1.6x lift.** Precision@25 = 0.480 (1.3x)
and Precision@50 = 0.540 (1.4x). The lift is real but modest, and it's worth being honest
about *why*: ML-06 found `avg_position_90d` and `active_days_90d` both run OPPOSITE to this
rule's assumed direction (better rank and more consistent visibility predict *more* decline,
not less), and `momentum_pct` alone tested FALSE (no meaningful univariate swing). So most
of this composite's lift is likely coming from the `visibility_score` gate itself -- bigger,
more-visible pages are simply more likely to be the ones with room to decline 30%+ in a
month -- rather than from the risk signals being individually well-aimed. That's a genuine
weakness for the Week 5 model to try to beat, not just match.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
WHAT_WOULD_MAKE_IT_WRONG = {
    'visible_declining_momentum': "momentum_pct is noisy for pages with a small Jan-Feb base -- a tiny baseline can swing the % a lot without a real problem.",
    'visible_poor_rank': "a poor average position can reflect a handful of bad-but-rare queries dragging the mean down, not the page's typical ranking.",
    'inconsistent_visibility': "low active_days_90d can mean the page is genuinely new, not declining -- check content_age separately if available.",
    'low_ctr_visible_page': "CTR is sensitive to SERP features (featured snippets, ads) outside the page's control.",
    'low_engagement_visible_page': "engagement ratio can be dragged down by a single traffic spike from an unrelated source.",
    'not_visible_low_priority': "a page with low impressions_90d might be brand new and simply hasn't accumulated visibility yet.",
    'general_review_candidate': "no specific risk fired -- this pick is riding on visibility alone, the weakest justification in the rule.",
}

def confidence_note(reasons):
    n_signals = len([r for r in reasons if r not in ('general_review_candidate', 'not_visible_low_priority')])
    if n_signals >= 2:
        return 'high (multiple independent risk signals agree)'
    if n_signals == 1:
        return 'medium (one risk signal)'
    return 'low (visibility only, no specific risk signal)'

top20 = dataset.sort_values('baseline_rank').head(20).copy()
top20['confidence'] = top20['reason_code_list'].apply(confidence_note)
top20['what_would_make_it_wrong'] = top20['reason_code_list'].apply(
    lambda rs: ' | '.join(sorted({WHAT_WOULD_MAKE_IT_WRONG[r] for r in rs}))
)

for _, row in top20.iterrows():
    print(f"#{row['baseline_rank']:<3} score={row['baseline_action_score']:.3f}  "
          f"action={row['suggested_action_baseline']:<26} actually_declining={bool(row['is_declining_label'])}")
    print(f"     reasons: {row['reason_codes']}")
    print(f"     confidence: {row['confidence']}")
    print(f"     what would make this wrong: {row['what_would_make_it_wrong']}")
    print()


#1   score=0.774  action=review                     actually_declining=False
     reasons: visible_declining_momentum|visible_poor_rank
     confidence: high (multiple independent risk signals agree)
     what would make this wrong: a poor average position can reflect a handful of bad-but-rare queries dragging the mean down, not the page's typical ranking. | momentum_pct is noisy for pages with a small Jan-Feb base -- a tiny baseline can swing the % a lot without a real problem.

#2   score=0.770  action=refresh_metadata_or_content actually_declining=False
     reasons: visible_declining_momentum|visible_poor_rank|low_engagement_visible_page
     confidence: high (multiple independent risk signals agree)
     what would make this wrong: a poor average position can reflect a handful of bad-but-rare queries dragging the mean down, not the page's typical ranking. | engagement ratio can be dragged down by a single traffic spike from an unrelated source. | momentum_pct is noisy for pages wi

## 3. Top-20 review

For each of the top 20, the cell above prints the rank, score, suggested action, reason
codes, a rule-based confidence note (more independent reason codes -> higher confidence),
whether the row is *actually* labeled declining (shown for review only -- never fed into the
score), and a caution specific to each reason code that fired.

**Pushing back on #1** (score 0.774, `visible_declining_momentum|visible_poor_rank`,
"high confidence", but `actually_declining=False`): this pick is confident for the wrong
reason. Its `visible_poor_rank` signal is exactly the one ML-06 found runs OPPOSITE to the
rule's assumption -- pages in the worst rank quartile are *less* likely to decline in this
dataset, not more -- so this "independent risk signal" is actively pointing the wrong way,
and "high confidence" here means "two votes, one of them backwards" rather than genuine
agreement. #8, #11, #17, and #20 are a different kind of weak pick: each is "medium
confidence" on `visible_declining_momentum` alone, a signal ML-06 already found has no
reliable univariate effect (FALSE verdict) -- so a single-signal medium-confidence pick from
this rule is closer to "visible and slightly above-median on a coin-flip signal" than a
genuine risk flag.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
import inspect

print("=== LEAKAGE CHECK ===")
leak_cols = ['impressions_mar', 'impressions_apr', 'pct_change', 'trend_direction', 'trend_pct']
src = inspect.getsource(reason_codes) + inspect.getsource(suggested_action)
score_inputs = ['impressions_90d', 'momentum_pct', 'avg_position_90d', 'active_days_90d',
                'ctr_90d', 'engaged_sessions_90d', 'sessions_90d', 'has_ga4_data', 'has_momentum']
leaked = [c for c in leak_cols if c in src]
print(f"Reason-code / action function source scanned for the {len(leak_cols)} ML-04 leak columns.")
print(f"Leak columns referenced: {leaked if leaked else 'none'}")
print(f"Columns the score actually reads: {score_inputs}")
print(f"Result: {'LEAKAGE DETECTED -- FIX BEFORE SUBMITTING' if leaked else 'No leakage -- rule only reads feature-window columns.'}")

print("\n=== WEAK PICKS ===")
weak = top20[top20['is_declining_label'] == 0]
print(f"Top-20 picks that are NOT actually declining: {len(weak)} of 20")
if len(weak):
    print(weak[['baseline_rank', 'reason_codes', 'baseline_action_score']].to_string(index=False))

top50_precision = dataset.sort_values('baseline_rank').head(50)['is_declining_label'].mean()
print(f"\nTop-50 false-positive rate: {1 - top50_precision:.1%} "
      f"(i.e. Precision@50 = {top50_precision:.3f}, matches section 2's printed value)")


=== LEAKAGE CHECK ===
Reason-code / action function source scanned for the 5 ML-04 leak columns.
Leak columns referenced: none
Columns the score actually reads: ['impressions_90d', 'momentum_pct', 'avg_position_90d', 'active_days_90d', 'ctr_90d', 'engaged_sessions_90d', 'sessions_90d', 'has_ga4_data', 'has_momentum']
Result: No leakage -- rule only reads feature-window columns.

=== WEAK PICKS ===
Top-20 picks that are NOT actually declining: 11 of 20
 baseline_rank                                                             reason_codes  baseline_action_score
             1                             visible_declining_momentum|visible_poor_rank               0.774042
             2 visible_declining_momentum|visible_poor_rank|low_engagement_visible_page               0.770239
             4                             visible_declining_momentum|visible_poor_rank               0.754087
             8                                               visible_declining_momentum             


Top-50 false-positive rate: 46.0% (i.e. Precision@50 = 0.540, matches section 2's printed value)


## 4. Weak picks + leakage check

The leakage check above programmatically scans the actual `reason_codes`/`suggested_action`
function source for every ML-04 excluded column (`impressions_mar`, `impressions_apr`,
`pct_change`, `trend_direction`, `trend_pct`) rather than asserting it by eye -- none of them
are even present in `work/outputs/dataset.csv` per the data contract, so this should always
come back clean, but the check is what makes that a verified fact instead of an assumption.

The weak-picks table lists any top-20 row that scored highly but isn't actually labeled
declining -- these are the rule's honest false positives, the same rows a content editor
would flag as "why did this show up here?" during a real review.

**11 of the top 20 (55%) are false positives, and the top-50 false-positive rate is 46.0%
(Precision@50 = 0.540).** Every single one of the 11 weak picks carries
`visible_declining_momentum`, and 8 of the 11 also carry `visible_poor_rank` -- the exact two
reason codes ML-06's real-data audit flagged as unreliable (momentum: FALSE, no meaningful
univariate swing; rank: OPPOSITE, running backwards from the rule's assumption). That's not
a coincidence: the weak picks aren't random noise, they're concentrated on the two reason
codes this rule leans on most heavily (0.45 + 0.30 = 75% of the risk-composite weight) that
ML-06 already showed don't hold up on their own. A model that replaces these two hand-picked
signals with ones the audit actually confirmed should be able to beat this baseline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.